# Preprocesamiento fMRI por lote — 46 sujetos (Unidad 3)
Orden: *(slice timing omitido: PRESTO/3D, sin información de cortes)* → realineación → corregistro BOLD→T1 → normalización T1→MNI (SyN) → suavizado 6 mm FWHM.
Cada paso guarda en disco y es **reanudable**: si ya existe, se salta. Ejecuta las celdas en orden.

## 0. Diagnóstico del entorno (ejecútala primero)

In [1]:
import sys, platform
from pathlib import Path

print("Python del kernel :", sys.version.split()[0])
print("Ejecutable        :", sys.executable)

# ¿Está ants instalado en ESTE kernel?
try:
    import ants
    print("ants OK ->", ants.__version__)
except ModuleNotFoundError:
    print("ants NO está instalado en este kernel.")
    # Busca carpetas 'ants' en tu proyecto para saber dónde quedó instalado
    raiz = Path(r"C:\fMRI")
    if raiz.exists():
        hallazgos = [p for p in raiz.rglob("ants") if p.is_dir() and p.parent.name == "site-packages"]
        for p in hallazgos:
            print("  encontrado en:", p.parent)
        if not hallazgos:
            print("  no hay ninguna instalación de ants bajo", raiz)

Python del kernel : 3.11.9
Ejecutable        : c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\python.exe
ants OK -> 0.6.3


**Si dice "ants NO está instalado"**, instálalo en el propio kernel (así se evita copiar `site-packages` de otro entorno, que falla porque los binarios de ANTs dependen de la versión de Python). Ejecuta la celda siguiente una sola vez y luego **reinicia el kernel** (Kernel → Restart):

In [2]:
# Descomenta y ejecuta UNA vez, luego reinicia el kernel
%pip install antspyx nibabel scipy matplotlib pandas requests

  Using cached antspyx-0.6.3-cp311-cp311-win_amd64.whl.metadata (7.2 kB)
  Using cached nibabel-5.4.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached matplotlib-3.11.2-cp311-cp311-win_amd64.whl.metadata (80 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached numpy-2.3.5-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached statsmodels-0.15.0-cp311-cp311-win_amd64.whl.metadata (9.6 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached scikit_learn-1.9.1-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.15.3-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Rutas del proyecto y de tu taller (`src`)

In [2]:
import sys
from pathlib import Path

ruta_taller = Path(r"C:\fMRI\Worskshop-fMRI-preprocessing-basics")   # contiene tu carpeta src
if str(ruta_taller) not in sys.path:
    sys.path.insert(0, str(ruta_taller))          # solo para poder importar 'src'

# OJO: NO agregamos el site-packages de otro venv; las librerias salen del kernel actual.
PROJECT_ROOT = Path.cwd().resolve()
print("Directorio de trabajo:", PROJECT_ROOT)

Directorio de trabajo: C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3


## 2. Importaciones y parámetros

In [12]:
import gc, gzip, os, shutil, subprocess, time, traceback

import ants
import matplotlib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

print("ants", ants.__version__, "| nibabel", nib.__version__)

FWHM_MM = 6.0
SIGMA_MM = FWHM_MM / (2.0 * np.sqrt(2.0 * np.log(2.0)))   # FWHM / 2.355
NORM_TRANSFORM = "SyN"       # mas rapido: "antsRegistrationSyNQuick[s]"

ants 0.6.3 | nibabel 5.4.2


In [3]:
# ======== EDITA ESTAS RUTAS ========
CSV_SUJETOS = Path(r"C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\sujetos_46.csv")
DATA_DIR    = Path(r"C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\ds005366_subset")
OUT_DIR     = Path(r"C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\derivatives_unidad3")
MNI_PATH    = Path(r"C:\fMRI\Worskshop-fMRI-preprocessing-basics\data\external\templates\MNI152_T1_2mm.nii.gz")
BORRAR_INTERMEDIOS = True    # borra el 4D realineado al terminar cada sujeto (ahorra disco)
# ===================================

for nombre, ruta in [("CSV", CSV_SUJETOS), ("DATA", DATA_DIR), ("MNI", MNI_PATH)]:
    print(f"{nombre:5s} {'OK ' if ruta.exists() else 'NO EXISTE'} {ruta}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV   OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\sujetos_46.csv
DATA  OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\ds005366_subset
MNI   OK  C:\fMRI\Worskshop-fMRI-preprocessing-basics\data\external\templates\MNI152_T1_2mm.nii.gz


## 3. Funciones (utilidades y pasos)

In [4]:
def liberar():
    gc.collect()

def ruta_existente(base: Path):
    """Acepta .nii o .nii.gz."""
    for cand in (base, Path(str(base) + ".gz")):
        if cand.exists():
            return cand
    raise FileNotFoundError(base)

def extraer_craneo(t1_path: Path, out: Path):
    """BET de FSL si esta en el PATH; si no, se usa la T1 completa (con aviso).
    La plantilla MNI152_T1_2mm de FSL conserva craneo, asi que el registro funciona igual."""
    if out.exists():
        return out
    bet = shutil.which("bet")
    if bet:
        subprocess.run([bet, str(t1_path), str(out), "-f", "0.4", "-R"], check=True)
        return out
    print("   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).")
    return t1_path

def guardar_transformaciones(res: dict, carpeta: Path, prefijo: str):
    """Copia a una ruta estable las transformaciones que ANTs deja en temporales."""
    carpeta.mkdir(parents=True, exist_ok=True)
    nuevas = []
    for i, f in enumerate(res["fwdtransforms"]):
        f = Path(f)
        sufijo = ".mat" if f.suffix == ".mat" else "".join(f.suffixes)
        destino = carpeta / f"{prefijo}_{i}{sufijo}"
        if f.resolve() != destino.resolve():
            shutil.copy(f, destino)
        nuevas.append(str(destino))
    return nuevas

def lista_transformaciones(carpeta: Path, prefijo: str):
    """Reconstruye la lista (orden de ANTs) de transformaciones guardadas."""
    return [str(p) for p in sorted(carpeta.glob(f"{prefijo}_*"))]

In [5]:
def paso_realineacion(bold_path: Path, d: Path):
    """Realineacion rigida 6 gl. Guarda: BOLD corregido, imagen media, parametros de movimiento."""
    moco_p, mean_p, mot_p = d / "step2_moco.nii.gz", d / "bold_mean.nii.gz", d / "motion.csv"
    if moco_p.exists() and mean_p.exists() and mot_p.exists():
        print("   [2] realineacion: ya existe, se salta")
        return moco_p, mean_p
    print("   [2] realineacion (Rigid)...")
    liberar()
    img = ants.image_read(str(bold_path))
    res = ants.motion_correction(img, type_of_transform="Rigid")
    moco = res["motion_corrected"]
    ants.image_write(moco, str(moco_p))

    # FD (si la version de ANTsPy lo entrega) para el control de calidad
    fd = res.get("FD", None)
    if fd is not None:
        pd.DataFrame({"FD": np.asarray(fd, dtype=float).ravel()}).to_csv(mot_p, index=False)
    else:
        pd.DataFrame({"FD": [np.nan]}).to_csv(mot_p, index=False)

    arr_mean = moco.numpy().mean(axis=3).astype(np.float32)
    mean = ants.from_numpy(
        arr_mean, origin=moco.origin[:3], spacing=moco.spacing[:3], direction=moco.direction[:3, :3]
    )
    ants.image_write(mean, str(mean_p))
    del img, res, moco, arr_mean, mean
    liberar()
    return moco_p, mean_p

In [6]:
def paso_registros(t1_path: Path, mean_p: Path, mni_path: Path, d: Path):
    """Corregistro BOLD->T1 (rigido) y normalizacion T1->MNI (SyN). Guarda transformaciones."""
    tdir = d / "transforms"
    if lista_transformaciones(tdir, "coreg") and lista_transformaciones(tdir, "norm"):
        print("   [3-4] registros: ya existen, se saltan")
        return
    print("   [3] corregistro BOLD -> T1 (Rigid, Mattes)...")
    liberar()
    t1 = ants.image_read(str(t1_path))
    mean = ants.image_read(str(mean_p))
    coreg = ants.registration(fixed=t1, moving=mean, type_of_transform="Rigid", aff_metric="mattes")
    guardar_transformaciones(coreg, tdir, "coreg")
    del coreg, mean
    liberar()

    print(f"   [4] normalizacion T1 -> MNI ({NORM_TRANSFORM})...")
    mni = ants.image_read(str(mni_path))
    norm = ants.registration(
        fixed=mni, moving=t1, type_of_transform=NORM_TRANSFORM, aff_metric="mattes", syn_metric="mattes"
    )
    guardar_transformaciones(norm, tdir, "norm")
    del norm, t1, mni
    liberar()

In [7]:
def paso_normaliza_suaviza(moco_p: Path, mni_path: Path, tr: float, d: Path, nombre: str):
    """Aplica coreg+norm (un solo remuestreo por volumen) y suaviza 6 mm FWHM.
    Escribe en streaming el .nii.gz final: un volumen en RAM a la vez."""
    final = d / f"{nombre}_space-MNI152_desc-preproc_bold.nii.gz"
    if final.exists():
        print("   [4-5] normalizacion+suavizado: ya existe, se salta")
        return final
    print("   [4-5] normalizando y suavizando volumen a volumen (streaming)...")
    tdir = d / "transforms"
    # ANTs aplica la lista de ultimo a primero: coreg (BOLD->T1) primero y luego norm (T1->MNI)
    transformlist = lista_transformaciones(tdir, "norm") + lista_transformaciones(tdir, "coreg")

    mni = ants.image_read(str(mni_path))
    mni_nii = nib.load(str(mni_path))
    affine = mni_nii.affine
    nx, ny, nz = mni.shape
    esp = np.array(mni.spacing[:3])
    sigma_vox = SIGMA_MM / esp

    moco_nii = nib.load(str(moco_p))          # carga perezosa
    nt = moco_nii.shape[3]
    ref = ants.image_read(str(d / "bold_mean.nii.gz"))  # geometria 3D nativa

    hdr = nib.Nifti1Header()
    hdr.set_data_dtype(np.float32)
    hdr.set_data_shape((nx, ny, nz, nt))
    hdr.set_zooms((*[float(e) for e in esp], float(tr)))
    hdr.set_xyzt_units("mm", "sec")
    hdr.set_sform(affine, code=4)
    hdr.set_qform(affine, code=4)
    hdr["vox_offset"] = 352

    suma = np.zeros((nx, ny, nz), dtype=np.float64)
    tmp = final.with_suffix("").with_suffix(".part.gz")
    t0 = time.time()
    with gzip.open(str(tmp), "wb", compresslevel=3) as f:
        hdr.write_to(f)
        pad = 352 - f.tell()
        if pad > 0:
            f.write(b"\x00" * pad)
        for i in range(nt):
            vol = np.asarray(moco_nii.dataobj[..., i], dtype=np.float32)
            vol_ants = ants.from_numpy(vol, origin=ref.origin, spacing=ref.spacing, direction=ref.direction)
            vol_mni = ants.apply_transforms(
                fixed=mni, moving=vol_ants, transformlist=transformlist,
                interpolator="linear", singleprecision=True,
            )
            arr = vol_mni.numpy().astype(np.float32)
            arr = gaussian_filter(arr, sigma=sigma_vox, mode="constant")
            suma += arr
            f.write(np.ascontiguousarray(arr).ravel(order="F").tobytes())  # orden NIfTI
            del vol, vol_ants, vol_mni, arr
            if (i + 1) % 50 == 0:
                liberar()
                print(f"      {i + 1}/{nt} volumenes ({(time.time() - t0) / 60:.1f} min)")
    tmp.replace(final)

    # media en MNI (util para QC y para el informe)
    nib.save(nib.Nifti1Image((suma / nt).astype(np.float32), affine), str(d / f"{nombre}_mean_mni.nii.gz"))
    del suma, mni, ref, moco_nii
    liberar()
    return final

In [8]:
def paso_qc(d: Path, mni_path: Path, nombre: str):
    """PNG de control: media funcional en MNI con contorno de la plantilla."""
    png = d / f"{nombre}_qc.png"
    mean_p = d / f"{nombre}_mean_mni.nii.gz"
    if png.exists() or not mean_p.exists():
        return
    m = nib.load(str(mean_p)).get_fdata()
    t = nib.load(str(mni_path)).get_fdata()
    cortes = [("Sagital", m.shape[0] // 2, 0), ("Coronal", m.shape[1] // 2, 1), ("Axial", m.shape[2] // 2 + 5, 2)]
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    vmax = np.percentile(m, 99)
    for a, (n, k, eje) in zip(ax, cortes):
        sl = lambda x: np.take(x, k, axis=eje).T
        a.imshow(sl(m), cmap="gray", origin="lower", vmin=0, vmax=vmax)
        a.contour(sl(t), levels=[np.percentile(t, 85)], colors="r", linewidths=0.6)
        a.set_title(n)
        a.axis("off")
    fig.suptitle(nombre)
    fig.tight_layout()
    fig.savefig(png, dpi=110)
    plt.close(fig)
    del m, t
    liberar()

In [9]:
def procesar_sujeto(fila: dict, data: Path, out: Path, mni_path: Path, borrar_intermedios: bool):
    sub, ses = fila["sub"], fila["ses"]
    nombre = f"{sub}_ses-{ses}_task-Motor2Class_run-{int(fila['run'])}"
    d = out / sub
    d.mkdir(parents=True, exist_ok=True)
    base = data / sub / f"ses-{ses}"

    bold_path = ruta_existente(base / fila["bold_file"])
    t1_raw = ruta_existente(base / fila["t1_file"])

    print(f"[{sub}] {fila['grupo']} | TR={fila['TR']} | {int(fila['NVol'])} vol")
    print("   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)")
    t1_path = extraer_craneo(t1_raw, d / "t1_brain.nii.gz")
    moco_p, mean_p = paso_realineacion(bold_path, d)
    paso_registros(t1_path, mean_p, mni_path, d)
    final = paso_normaliza_suaviza(moco_p, mni_path, float(fila["TR"]), d, nombre)
    paso_qc(d, mni_path, nombre)

    if borrar_intermedios and final.exists() and moco_p.exists():
        moco_p.unlink()  # el 4D realineado ya no hace falta
    return final

## 4. Prueba con 1–2 sujetos (recomendado antes del lote)

In [13]:
import struct, shutil

bold = DATA_DIR / "sub-083/ses-3T/func/sub-083_ses-3T_task-Motor2Class_run-3_bold.nii"
print("Tamaño (MB):", round(bold.stat().st_size / 1e6, 1))

with open(bold, "rb") as fh:
    head = fh.read(348)
print("¿gzip disfrazado?:", head[:2] == b"\x1f\x8b")
print("sizeof_hdr:", struct.unpack("<i", head[:4])[0], "(NIfTI-1 = 348)")
print("magic:", head[344:348], "(NIfTI-1 = b'n+1\\x00')")

img = nib.load(str(bold))                      # nibabel sí maneja rutas con acentos
print("Nibabel lee OK, forma:", img.shape)

# Prueba decisiva: copia a una ruta sin acentos y que ANTs la lea
tmp = Path(r"C:\fMRI\_prueba_ascii.nii")
shutil.copy(bold, tmp)
try:
    ants.image_read(str(tmp))
    print(">> ANTs lee la copia en ruta ASCII: el problema es la RUTA con acentos.")
except Exception as e:
    print(">> ANTs también falla en ruta ASCII:", e)
tmp.unlink()

Tamaño (MB): 62.9
¿gzip disfrazado?: False
sizeof_hdr: 348 (NIfTI-1 = 348)
magic: b'n+1\x00' (NIfTI-1 = b'n+1\x00')
Nibabel lee OK, forma: (80, 80, 40, 120)
>> ANTs lee la copia en ruta ASCII: el problema es la RUTA con acentos.


In [14]:
df = pd.read_csv(CSV_SUJETOS)
print(df.groupby(["grupo", "sex"]).size())

def correr(subs):
    res = []
    for _, r in df[df["sub"].isin(subs)].iterrows():
        t0 = time.time()
        try:
            f = procesar_sujeto(r.to_dict(), DATA_DIR, OUT_DIR, MNI_PATH, BORRAR_INTERMEDIOS)
            res.append(dict(sub=r["sub"], grupo=r["grupo"], estado="ok", minutos=round((time.time()-t0)/60, 1), error=""))
        except Exception as e:
            traceback.print_exc()
            res.append(dict(sub=r["sub"], grupo=r["grupo"], estado="ERROR", minutos=round((time.time()-t0)/60, 1), error=repr(e)))
        finally:
            liberar()
    return res

# prueba: un sano y un paciente
prueba = [df[df.grupo == "sano"]["sub"].iloc[0], df[df.grupo == "epilepsia"]["sub"].iloc[0]]
res_prueba = correr(prueba)
pd.DataFrame(res_prueba)

grupo      sex
epilepsia  F       8
           M       8
sano       F      15
           M      15
dtype: int64
[sub-083] sano | TR=2.5 | 120 vol
   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)
   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).
   [2] realineacion (Rigid)...
   [3] corregistro BOLD -> T1 (Rigid, Mattes)...
   [4] normalizacion T1 -> MNI (SyN)...
   [4-5] normalizando y suavizando volumen a volumen (streaming)...
      50/120 volumenes (0.6 min)
      100/120 volumenes (1.6 min)
[sub-009] epilepsia | TR=0.61 | 690 vol
   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)
   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).
   [2] realineacion (Rigid)...
   [3] corregistro BOLD -> T1 (Rigid, Mattes)...
   [4] normalizacion T1 -> MNI (SyN)...
   [4-5] normalizando y suavizando volumen a volumen (streaming)...
      50/690 volumenes (0.5 min)
      100/690 volumenes (1.4 min

,sub,grupo,estado,minutos,error
0,sub-083,sano,ok,7.0,
1,sub-009,epilepsia,ok,41.3,


## 5. Lote completo (46 sujetos, secuencial y reanudable)

In [15]:
log_path = OUT_DIR / "preproc_log.csv"
resultados = []
for s in df["sub"]:
    resultados += correr([s])
    pd.DataFrame(resultados).to_csv(log_path, index=False)   # log se actualiza sujeto a sujeto

res = pd.DataFrame(resultados)
print(res.estado.value_counts())
res

[sub-083] sano | TR=2.5 | 120 vol
   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)
   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).
   [2] realineacion (Rigid)...
   [3-4] registros: ya existen, se saltan
   [4-5] normalizacion+suavizado: ya existe, se salta
[sub-084] sano | TR=2.5 | 120 vol
   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)
   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).
   [2] realineacion (Rigid)...
   [3] corregistro BOLD -> T1 (Rigid, Mattes)...
   [4] normalizacion T1 -> MNI (SyN)...
   [4-5] normalizando y suavizando volumen a volumen (streaming)...
      50/120 volumenes (0.5 min)
      100/120 volumenes (1.3 min)
[sub-087] sano | TR=2.5 | 168 vol
   [1] slice timing: omitido (PRESTO/3D; sin informacion de cortes)
   [aviso] FSL 'bet' no encontrado: se registra la T1 completa (sin extraer craneo).
   [2] realineacion (Rigid)...
   [3] corregistro BOLD ->

,sub,grupo,estado,minutos,error
0,sub-083,sano,ok,2.8,
1,sub-084,sano,ok,5.8,
2,sub-087,sano,ok,8.0,
3,sub-090,sano,ok,7.7,
4,sub-091,sano,ok,7.7,
5,sub-094,sano,ok,5.5,
6,sub-095,sano,ok,7.4,
7,sub-101,sano,ok,7.7,
8,sub-102,sano,ok,7.5,
9,sub-103,sano,ok,5.7,


In [2]:
import nibabel as nib, pandas as pd, numpy as np
from pathlib import Path

# El notebook está en Proyecto_U3, así que las rutas salen de la carpeta actual
RAIZ        = Path.cwd().resolve()
CSV_SUJETOS = RAIZ / "sujetos_46.csv"
OUT_DIR     = RAIZ / "derivatives_unidad3"

for nombre, ruta in [("RAIZ", RAIZ), ("CSV", CSV_SUJETOS), ("OUT", OUT_DIR)]:
    print(f"{nombre:5s} {'OK ' if ruta.exists() else 'NO EXISTE'} {ruta}")

df = pd.read_csv(CSV_SUJETOS)
VERIFICAR_INTEGRIDAD = False   # True = lee el último volumen (más lento, detecta gz truncados)

filas = []
for _, r in df.iterrows():
    nombre = f"{r['sub']}_ses-{r['ses']}_task-Motor2Class_run-{int(r['run'])}"
    f = OUT_DIR / r["sub"] / f"{nombre}_space-MNI152_desc-preproc_bold.nii.gz"
    estado, nvol = "FALTA", None
    if f.exists():
        try:
            img = nib.load(str(f))
            nvol = img.shape[3] if len(img.shape) == 4 else None
            estado = "ok" if nvol == int(r["NVol"]) else f"NVol distinto ({nvol} vs {int(r['NVol'])})"
            if VERIFICAR_INTEGRIDAD and estado == "ok":
                _ = np.asarray(img.dataobj[..., -1])
        except Exception as e:
            estado = f"CORRUPTO: {e!r}"
    filas.append(dict(sub=r["sub"], grupo=r["grupo"], estado=estado, nvol=nvol,
                      GB=round(f.stat().st_size / 1e9, 2) if f.exists() else None,
                      qc_png=any((OUT_DIR / r["sub"]).glob("*_qc.png"))))
ver = pd.DataFrame(filas)
print(ver.estado.value_counts())
ver[ver.estado != "ok"]

RAIZ  OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3
CSV   OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\sujetos_46.csv
OUT   OK  C:\Users\Lenovo\Documents\Paulina 9S\P Imagenes\Proyecto_U3\derivatives_unidad3
estado
ok    46
Name: count, dtype: int64


,sub,grupo,estado,nvol,GB,qc_png


## 6. Revisión rápida de calidad (FD por sujeto)

In [3]:
filas = []
for s in df["sub"]:
    m = OUT_DIR / s / "motion.csv"
    if m.exists():
        fd = pd.read_csv(m)["FD"].dropna()
        filas.append(dict(sub=s, FD_medio=fd.mean(), FD_max=fd.max(), pct_FD_gt_1mm=(fd > 1).mean() * 100))
qc = df[["sub", "grupo", "sex", "age"]].merge(pd.DataFrame(filas), on="sub", how="left")
qc.to_csv(OUT_DIR / "qc_movimiento.csv", index=False)
qc.sort_values("FD_medio", ascending=False).head(10)

,sub,grupo,sex,age,FD_medio,FD_max,pct_FD_gt_1mm
31,sub-019,epilepsia,F,14,0.517327,2.472229,6.206897
35,sub-025,epilepsia,F,19,0.354867,1.846954,3.167421
34,sub-023,epilepsia,F,47,0.262405,0.706959,0.000000
36,sub-030,epilepsia,M,13,0.234137,0.809237,0.000000
44,sub-076,epilepsia,M,25,0.217444,0.678459,0.000000
43,sub-072,epilepsia,M,11,0.206935,1.570017,1.131222
41,sub-066,epilepsia,F,50,0.205949,0.962547,0.000000
45,sub-155,epilepsia,M,16,0.203929,0.587896,0.000000
30,sub-009,epilepsia,F,17,0.200131,0.859636,0.000000
32,sub-020,epilepsia,F,15,0.158771,0.393921,0.000000
